In [1]:
import os
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\pneumonia-segmentation'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class LLMParams:
    model:      str
    max_tokens: int

@dataclass(frozen=True)
class ClaudeArchitectureConfig:
    root_dir:           Path
    history_path:       Path
    prometheus_url:     str
    orchestrator_url:   str
    params:         LLMParams

In [3]:
from core.constants import *
from core.utils import read_yaml, create_directories

class ConfigurationManger:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_claude_architecture_config(self) -> ClaudeArchitectureConfig:
        config = self.config.claude_config
        params = self.params.claude_params
        create_directories([config.root_dir])
        
        return ClaudeArchitectureConfig(
            root_dir     = Path(config.root_dir),
            history_path = Path(config.root_dir) / "architecture_history.json",
            prometheus_url   = params.prometheus_url,
            orchestrator_url = params.orchestrator_url,
            params = LLMParams(
                model        = params.model,
                max_tokens   = params.max_tokens,
            )
        )

In [4]:
import json, time, os, sys, httpx, anthropic
from core.logging import logger
from core.exception import CustomException
from dotenv import load_dotenv

load_dotenv()

class ClaudeArchitecture:
    def __init__(self, config: ClaudeArchitectureConfig):
        self.config  = config
        self.client  = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        self.history = self._load_history()
    
    def _load_history(self) -> list:
        history_path = self.config.history_path
        
        history_path.parent.mkdir(parents=True, exist_ok=True)
        if history_path.exists():
            return json.loads(history_path.read_text())
        return []
        
    def _get_topology_snapshot(self) -> dict:
        q = self._query_prometheus
        return {
            "services": {
                "app": {
                    "cpu":     q('rate(process_cpu_seconds_total{job="app"}[5m]) * 100'),
                    "ram":     q('process_resident_memory_bytes{job="app"} / 1024 / 1024'),
                    "latency": q('histogram_quantile(0.95, rate(service_request_latency_seconds_bucket{job="app", endpoint="/predict"}[5m]))'),
                    "drift":   q('inference_drift_score{job="app"}'),
                },
                "lstm": {
                    "cpu": q('rate(process_cpu_seconds_total{job="lstm"}[5m]) * 100'),
                    "ram": q('process_resident_memory_bytes{job="lstm"} / 1024 / 1024'),
                },
                "dqn": {
                    "cpu": q('rate(process_cpu_seconds_total{job="dqn"}[5m]) * 100'),
                    "ram": q('process_resident_memory_bytes{job="dqn"} / 1024 / 1024'),
                }
            }
        }
    
    def _query_prometheus(self, query: str) -> float:
        try: 
            r = httpx.get(
                f"{self.config.prometheus_url}/api/v1/query", 
                params={"query": query}
            )
            
            result = r.json()["data"]["result"]
            return float(result[0]["value"][1]) if result else 0.0
        except Exception as e:
            return 0.0
    
    
    def get_claude_decision(self) -> dict:
        try:
            topology  = self._get_topology_snapshot()
            
            response  = self.client.messages.create(
                model = self.config.params.model,
                max_tokens = self.config.params.max_tokens,
                messages = [{
                    "role": "user",
                    "content": self._build_prompt(topology)
                }]
            )       
        
            result = self._parse_response(response.content[0].text)
            self._save_history(result, topology)
            logger.info(f"Claude architect: {result['action']} — {result['reasoning']}")
            
            if result.get("evolution_needed") and result["action"] == "none":
                self._execute_evolution(result)
            
            return result
        except Exception as e:
            raise CustomException(e, sys)
        
    def _build_prompt(self, topology: dict) -> str:
        history_str = ""
        if self.history:
            history_str = f"\nPrevious architectural decisions:\n{json.dumps(self.history[-3:], indent=2)}\n"
        
    
        return f"""You are an AI system architect managing a medical image segmentation MLOps platform.
            {history_str}
            Current system topology:
            {json.dumps(topology, indent=2)}

            Analyze the system state and decide if architectural evolution is needed.

            Respond ONLY with a JSON object, no explanation, no markdown:
            {{
                "evolution_needed": true/false,
                "action": "none" | "spawn" | "swap" | "rollback",
                "target_service": "app" | "lstm" | "dqn" | "none",
                "parameters": {{}},
                "reasoning": "brief explanation",
                "confidence": 0.0-1.0
            }}

            Rules:
            - spawn: if app latency > 1.0s AND cpu > 70%
            - swap: if drift > 60 (on scale 0-100) AND current model is int8
            - rollback: if latency suddenly increased > 2x after recent action
            - none: if system is healthy
            """
    
    def _parse_response(self, raw: str) -> dict:
        cleaned = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(cleaned)

    def _save_history(self, result: dict, topology: dict):
        self.history.append({
            "timestamp": time.time(),
            "topology_summary": {
                "app_drift": topology["services"]["app"]["drift"],
                "app_cpu":   topology["services"]["app"]["cpu"],
                "app_ram":   topology["services"]["app"]["ram"],
            },
            "action":    result["action"],
            "reasoning": result["reasoning"][:150].encode('ascii', 'ignore').decode()
        })
        self.config.history_path.write_text(json.dumps(self.history[-10:], indent=2))
    
    def _execute_evolution(self, decision: dict) -> None:
        action_map = {
            "spawn":    "scale_out_service",
            "swap":     "swap_model_version",
            "rollback": "rollback"
        }
        
        orchestrator_action = action_map.get(decision["action"])
        if not orchestrator_action:
            return
        
        try: 
            res = httpx.post(
                f"{self.config.orchestrator_url}/execute/{orchestrator_action}",
                timeout=30
            )
            logger.info(f"Evolution executed: {orchestrator_action} → {res.json()}")
        except Exception as e:
            logger.error(f"Evolution execution failed: {e}")
        

In [5]:
try:
    config = ConfigurationManger()
    claude_config    = config.get_claude_architecture_config()
    claude_architect = ClaudeArchitecture(config=claude_config)
    results = claude_architect.get_claude_decision()
    
    print(results['action'])
    print(results['reasoning'])
except Exception as e:
    raise CustomException(e, sys)

[2026-04-30 18:23:50,724: INFO: __init__: yaml file: config\config.yaml loaded successfully]


[2026-04-30 18:23:50,739: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-04-30 18:23:50,741: INFO: __init__: created directory at: artifacts]
[2026-04-30 18:23:50,743: INFO: __init__: created directory at: artifacts/claude]
[2026-04-30 18:24:16,437: INFO: _client: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"]
[2026-04-30 18:24:16,464: INFO: 580589349: Claude architect: none — All services report zero metrics (cpu, ram, latency, drift). System appears to be in initialization or idle state. No thresholds are exceeded: latency 0.0s (below 1.0s spawn threshold), cpu 0.0% (below 70% spawn threshold), drift 0.0 (below 60 swap threshold). No recent action to trigger rollback evaluation.]
none
All services report zero metrics (cpu, ram, latency, drift). System appears to be in initialization or idle state. No thresholds are exceeded: latency 0.0s (below 1.0s spawn threshold), cpu 0.0% (below 70% spawn threshold), drift 0.0 (below 60 swap thresho

In [6]:
try:
    config = ConfigurationManger()
    claude_config    = config.get_claude_architecture_config()
    claude_architect = ClaudeArchitecture(config=claude_config)
    claude_architect._get_topology_snapshot = lambda: {
        "services": {
            "app": {"cpu": 85.0, "ram": 500.0, "latency": 1.5, "drift": 20},
            "lstm": {"cpu": 10.0, "ram": 200.0},
            "dqn":  {"cpu": 5.0,  "ram": 150.0}
        }
    }
    results = claude_architect.get_claude_decision()
    
    print(results['action'])
    print(results['reasoning'])
except Exception as e:
    raise CustomException(e, sys)

[2026-04-30 18:24:16,475: INFO: __init__: yaml file: config\config.yaml loaded successfully]
[2026-04-30 18:24:16,483: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-04-30 18:24:16,484: INFO: __init__: created directory at: artifacts]
[2026-04-30 18:24:16,486: INFO: __init__: created directory at: artifacts/claude]
[2026-04-30 18:24:18,878: INFO: _client: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"]
[2026-04-30 18:24:18,881: INFO: 580589349: Claude architect: spawn — App service triggers spawn condition: CPU at 85% (exceeds 70% threshold) and latency at 1.5s (exceeds 1.0s threshold). Drift at 20 is acceptable. System requires horizontal scaling to handle load.]
spawn
App service triggers spawn condition: CPU at 85% (exceeds 70% threshold) and latency at 1.5s (exceeds 1.0s threshold). Drift at 20 is acceptable. System requires horizontal scaling to handle load.


In [7]:
try:
    config = ConfigurationManger()
    claude_config    = config.get_claude_architecture_config()
    claude_architect = ClaudeArchitecture(config=claude_config)
    claude_architect._get_topology_snapshot = lambda: {
        "services": {
            "app": {"cpu": 20.0, "ram": 400.0, "latency": 0.1, "drift": 90.0},
            "lstm": {"cpu": 10.0, "ram": 200.0},
            "dqn":  {"cpu": 5.0,  "ram": 150.0}
        }
    }
    results = claude_architect.get_claude_decision()
    
    print(results['action'])
    print(results['reasoning'])
except Exception as e:
    raise CustomException(e, sys)

[2026-04-30 18:24:18,894: INFO: __init__: yaml file: config\config.yaml loaded successfully]
[2026-04-30 18:24:18,900: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-04-30 18:24:18,903: INFO: __init__: created directory at: artifacts]
[2026-04-30 18:24:18,904: INFO: __init__: created directory at: artifacts/claude]
[2026-04-30 18:24:20,907: INFO: _client: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"]
[2026-04-30 18:24:20,911: INFO: 580589349: Claude architect: swap — App service drift at 90.0 exceeds swap threshold of 60. Despite healthy CPU (20%) and latency (0.1s), high drift indicates model performance degradation requiring model replacement. Swap action recommended to deploy updated model.]
swap
App service drift at 90.0 exceeds swap threshold of 60. Despite healthy CPU (20%) and latency (0.1s), high drift indicates model performance degradation requiring model replacement. Swap action recommended to deploy updated model.


In [8]:
try:
    config = ConfigurationManger()
    claude_config    = config.get_claude_architecture_config()
    claude_architect = ClaudeArchitecture(config=claude_config)
    claude_architect._get_topology_snapshot = lambda: {
        "services": {
            "app": {"cpu": 20.0, "ram": 400.0, "latency": 0.1, "drift": 30},
            "lstm": {"cpu": 10.0, "ram": 200.0},
            "dqn":  {"cpu": 5.0,  "ram": 150.0}
        }
    }
    results = claude_architect.get_claude_decision()
    
    print(results['action'])
    print(results['reasoning'])
except Exception as e:
    raise CustomException(e, sys)

[2026-04-30 18:24:20,926: INFO: __init__: yaml file: config\config.yaml loaded successfully]
[2026-04-30 18:24:20,934: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-04-30 18:24:20,936: INFO: __init__: created directory at: artifacts]
[2026-04-30 18:24:20,937: INFO: __init__: created directory at: artifacts/claude]
[2026-04-30 18:24:22,736: INFO: _client: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"]
[2026-04-30 18:24:22,740: INFO: 580589349: Claude architect: none — System is healthy: app CPU at 20% (below 70% threshold), latency at 0.1s (below 1.0s threshold), and drift at 30 (below 60 swap threshold). Recent swap action at timestamp 1777548260.9 successfully reduced drift from 90 to 30. No thresholds exceeded.]
none
System is healthy: app CPU at 20% (below 70% threshold), latency at 0.1s (below 1.0s threshold), and drift at 30 (below 60 swap threshold). Recent swap action at timestamp 1777548260.9 successfully reduced drift from 90 to